# 03 - Dataset Pipeline Check - Jena Climate (Seq2Seq Weather Forecasting)

This notebook proves that `src/data/dataset.py` (`WeatherForecastDataset`)
and `src/data/dataloader.py` correctly turn the leakage-safe processed
splits from step 02 into (168h history -> 72h forecast) samples and
model-ready batches. It only **calls** the pipeline and inspects its
output - the sliding-window logic itself lives in the modules, not here.
No model is trained in this notebook.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd
import torch

import src.data.preprocessing as pp
from src.data.dataloader import build_dataloaders
from src.data.dataset import WeatherForecastDataset, validate_weather_batch

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "preprocessing"
print("Project root:", PROJECT_ROOT)
print("Processed dir exists:", PROCESSED_DIR.exists())

Project root: E:\Project_Deep\Project_DL_Weather_Forecasting
Processed dir exists: True


## 1. Load processed dataset

Reads the three leakage-safe splits produced by step 02
(`src/data/preprocessing.py`): already cleaned, resampled to hourly,
feature-engineered, chronologically split, imputed, and scaled.

In [2]:
train_df = pd.read_csv(PROCESSED_DIR / "train_processed.csv", parse_dates=[pp.TIMESTAMP_COLUMN])
val_df = pd.read_csv(PROCESSED_DIR / "val_processed.csv", parse_dates=[pp.TIMESTAMP_COLUMN])
test_df = pd.read_csv(PROCESSED_DIR / "test_processed.csv", parse_dates=[pp.TIMESTAMP_COLUMN])

print(f"train_processed.csv: {train_df.shape}")
print(f"val_processed.csv:   {val_df.shape}")
print(f"test_processed.csv:  {test_df.shape}")
train_df.head(3)

train_processed.csv: (49090, 33)
val_processed.csv:   (10519, 33)
test_processed.csv:  (10520, 33)


,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),...,missing_rh_pct,missing_VPmax_mbar,missing_VPact_mbar,missing_VPdef_mbar,missing_sh_gkg,missing_H2OC_mmolmol,missing_rho_gm3,missing_wv_ms,missing_max_wv_ms,missing_wd_deg
0,2009-01-01 00:00:00,0.948907,-2.014297,-2.074466,-1.968587,1.083659,-1.314739,-1.497932,-0.790657,-1.499869,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2009-01-01 01:00:00,0.948545,-1.986651,-2.046505,-1.931547,1.092974,-1.306490,-1.482811,-0.790727,-1.484279,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2009-01-01 02:00:00,0.975079,-2.067430,-2.128743,-2.038843,1.068674,-1.329717,-1.525958,-0.790727,-1.527308,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Load feature_schema.json

`feature_schema.json` is the locked contract from step 02: it fixes the
ordered feature list, the target column, and the window/horizon sizes so
they are never hardcoded here or in shared model code.

In [3]:
with open(ARTIFACTS_DIR / "feature_schema.json", encoding="utf-8") as f:
    schema = json.load(f)

FEATURES = schema["ordered_features"]
TARGET = schema["target"]["name"]
INPUT_WINDOW = schema["window_contract"]["input_len_hours"]
HORIZON = schema["window_contract"]["horizon_hours"]

print(f"n_features = {len(FEATURES)}")
print(f"target = {TARGET!r} (schema index {schema['target']['index']})")
print(f"input_window = {INPUT_WINDOW} hours, horizon = {HORIZON} hours")

assert FEATURES == pp.FEATURE_COLUMNS, "notebook schema drifted from src.data.preprocessing.FEATURE_COLUMNS"
assert TARGET == pp.TARGET_COLUMN
print("OK: schema matches the preprocessing module's contract constants.")

n_features = 32
target = 'T (degC)' (schema index 1)
input_window = 168 hours, horizon = 72 hours
OK: schema matches the preprocessing module's contract constants.


## 3. Create Dataset

One `WeatherForecastDataset` per split - never a concatenation of splits.

In [4]:
train_dataset = WeatherForecastDataset(train_df, FEATURES, TARGET, input_window=INPUT_WINDOW, horizon=HORIZON)
val_dataset = WeatherForecastDataset(val_df, FEATURES, TARGET, input_window=INPUT_WINDOW, horizon=HORIZON)
test_dataset = WeatherForecastDataset(test_df, FEATURES, TARGET, input_window=INPUT_WINDOW, horizon=HORIZON)

print("WeatherForecastDataset(\n"
      f"  df=train_df, features=<{len(FEATURES)} cols>, target={TARGET!r},\n"
      f"  input_window={INPUT_WINDOW}, horizon={HORIZON}\n)")

WeatherForecastDataset(
  df=train_df, features=<32 cols>, target='T (degC)',
  input_window=168, horizon=72
)


## 4. Dataset length and sample shape

Length is the number of *valid* sliding windows: raw row count minus
`input_window + horizon - 1`, minus any window that touches a gap where
`T (degC)` (or any feature) was left missing by preprocessing.

In [5]:
raw_windows = {
    "train": len(train_df) - (INPUT_WINDOW + HORIZON) + 1,
    "validation": len(val_df) - (INPUT_WINDOW + HORIZON) + 1,
    "test": len(test_df) - (INPUT_WINDOW + HORIZON) + 1,
}
kept_windows = {
    "train": len(train_dataset),
    "validation": len(val_dataset),
    "test": len(test_dataset),
}
summary = pd.DataFrame({"raw_candidate_windows": raw_windows, "kept_windows": kept_windows})
summary["dropped_for_missing_data"] = summary["raw_candidate_windows"] - summary["kept_windows"]
summary

,raw_candidate_windows,kept_windows,dropped_for_missing_data
train,48851,48851,0
validation,10280,10026,254
test,10281,9969,312


In [6]:
sample = train_dataset[0]
print("x:", sample["x"].shape, sample["x"].dtype)
print("y:", sample["y"].shape, sample["y"].dtype)
print("input_timestamps:", sample["input_timestamps"].shape, sample["input_timestamps"].dtype)
print("target_timestamps:", sample["target_timestamps"].shape, sample["target_timestamps"].dtype)

assert sample["x"].shape == (INPUT_WINDOW, len(FEATURES))
assert sample["y"].shape == (HORIZON, 1)
assert torch.isfinite(sample["x"]).all() and torch.isfinite(sample["y"]).all()
print("\nOK: sample shapes match [168, n_features] / [72, 1] and contain no NaN/Inf.")

x: torch.Size([168, 32]) torch.float32
y: torch.Size([72, 1]) torch.float32
input_timestamps: torch.Size([168]) torch.int64
target_timestamps: torch.Size([72]) torch.int64

OK: sample shapes match [168, n_features] / [72, 1] and contain no NaN/Inf.


## 5. Timeline check

Confirms the forecast period begins exactly one hour after the input
period ends - no gap, no overlap - for a sample from each split.

In [7]:
def describe_window(dataset: WeatherForecastDataset, name: str, index: int = 0) -> None:
    bounds = dataset.window_bounds(index)
    gap_hours = (bounds["forecast_start"] - bounds["input_end"]).total_seconds() / 3600
    print(f"[{name}] sample {index}")
    print(f"  input period:    {bounds['input_start']}  ->  {bounds['input_end']}")
    print(f"  forecast period: {bounds['forecast_start']}  ->  {bounds['forecast_end']}")
    print(f"  gap between input end and forecast start: {gap_hours:.0f}h (expect 1h)\n")
    assert gap_hours == 1.0, "forecast must start immediately after input ends"


describe_window(train_dataset, "train")
describe_window(val_dataset, "validation")
describe_window(test_dataset, "test")
print("OK: forecast starts exactly 1 hour after input ends in every split.")

[train] sample 0
  input period:    2009-01-01 00:00:00  ->  2009-01-07 23:00:00
  forecast period: 2009-01-08 00:00:00  ->  2009-01-10 23:00:00
  gap between input end and forecast start: 1h (expect 1h)

[validation] sample 0
  input period:    2014-08-08 10:00:00  ->  2014-08-15 09:00:00
  forecast period: 2014-08-15 10:00:00  ->  2014-08-18 09:00:00
  gap between input end and forecast start: 1h (expect 1h)

[test] sample 0
  input period:    2015-10-20 17:00:00  ->  2015-10-27 16:00:00
  forecast period: 2015-10-27 17:00:00  ->  2015-10-30 16:00:00
  gap between input end and forecast start: 1h (expect 1h)

OK: forecast starts exactly 1 hour after input ends in every split.


In [8]:
with open(ARTIFACTS_DIR / "split_metadata.json", encoding="utf-8") as f:
    split_metadata = json.load(f)

train_end = pd.Timestamp(split_metadata["train"]["end"])
val_start = pd.Timestamp(split_metadata["validation"]["start"])

last_train_bounds = train_dataset.window_bounds(len(train_dataset) - 1)
first_val_bounds = val_dataset.window_bounds(0)

print(f"Last train sample's forecast ends:   {last_train_bounds['forecast_end']}  (train split ends {train_end})")
print(f"First val sample's input starts:     {first_val_bounds['input_start']}  (validation split starts {val_start})")

assert last_train_bounds["forecast_end"] <= train_end
assert first_val_bounds["input_start"] >= val_start
print("\nOK: no train sample's window reaches into validation, and no validation sample reaches back into train.")

Last train sample's forecast ends:   2014-08-08 09:00:00  (train split ends 2014-08-08 09:00:00)
First val sample's input starts:     2014-08-08 10:00:00  (validation split starts 2014-08-08 10:00:00)

OK: no train sample's window reaches into validation, and no validation sample reaches back into train.


## 6. Batch check

Wrap each split in a DataLoader and confirm the batch dimension is added correctly.

In [9]:
BATCH_SIZE = 32
loaders = build_dataloaders(train_dataset, val_dataset, test_dataset, batch_size=BATCH_SIZE)

train_batch = next(iter(loaders["train"]))
print("X batch:", train_batch["x"].shape, " (expect [B, 168, n_features])")
print("y batch:", train_batch["y"].shape, " (expect [B, 72, 1])")
print("input_timestamps batch:", train_batch["input_timestamps"].shape)
print("target_timestamps batch:", train_batch["target_timestamps"].shape)

assert train_batch["x"].shape == (BATCH_SIZE, INPUT_WINDOW, len(FEATURES))
assert train_batch["y"].shape == (BATCH_SIZE, HORIZON, 1)

# Validate against TV1's locked WeatherBatch contract - the same check model code relies on.
validate_weather_batch(train_batch, n_features=len(FEATURES))
print("\nOK: batch shapes match [B,168,n_features] / [B,72,1] and pass validate_weather_batch.")

X batch: torch.Size([32, 168, 32])  (expect [B, 168, n_features])
y batch: torch.Size([32, 72, 1])  (expect [B, 72, 1])
input_timestamps batch: torch.Size([32, 168])
target_timestamps batch: torch.Size([32, 72])

OK: batch shapes match [B,168,n_features] / [B,72,1] and pass validate_weather_batch.


In [10]:
val_batch = next(iter(loaders["val"]))
test_batch = next(iter(loaders["test"]))
validate_weather_batch(val_batch, n_features=len(FEATURES))
validate_weather_batch(test_batch, n_features=len(FEATURES))
print("OK: validation and test batches also pass the contract validator.")

OK: validation and test batches also pass the contract validator.


## Summary

- `WeatherForecastDataset` builds 168h-in / 72h-out sliding windows
  independently per split, dropping any window that would touch a missing
  value - never fabricating training data.
- Windows never cross a split boundary because each dataset is built from
  only that split's rows.
- Batches produced by `src/data/dataloader.py` match the `WeatherBatch`
  contract (`x [B,168,F]`, `y [B,72,1]`, aligned timestamps) that
  `src/data/dataset.py::validate_weather_batch` enforces for every model
  (Seq2Seq LSTM, Attention LSTM, Transformer).
- The dataset pipeline is ready to hand off to the Model Training team.